# GinSign Grounder — Joint Model Evaluation

Loads the **single** BERT model trained jointly on predicate + argument shards and evaluates it on the **test** split with a breakdown:
- Overall metrics
- **Per domain** (`<search_and_rescue>`, `<warehouse>`, `<traffic_light>`) 
- **Per predicate** (tokens after `<predicates>`) 
- **Per constant** (tokens after `<const>`)

In [ ]:

# Install deps (safe to run multiple times)
import sys, subprocess, pkgutil
def _pip(pkg): 
    if pkgutil.find_loader(pkg.split("==")[0]) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
_pip("datasets")
_pip("transformers")
_pip("scikit-learn")
_pip("pandas")
_pip("accelerate")


In [ ]:

from pathlib import Path
from typing import List, Dict, Any, Tuple, Iterable
import os, json, math, numpy as np, pandas as pd

import torch
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# --- Config ---
DATA_ROOT = Path("libero_grounding_data")            # path produced by build_grounding_data.py
MODEL_DIR = Path("outputs_joint")             # folder where the single joint model was saved
BATCH_SIZE = 32
MAX_PREFIX_SHARD = 20                         # must match data builder
THRESH = 0.5

assert MODEL_DIR.exists(), f"Model dir not found: {MODEL_DIR}"


In [ ]:

# # -------------------------------
# # Data loading (test split only)
# # -------------------------------
# def _read_jsonl(paths: List[Path]) -> List[Dict[str, Any]]:
#     rows = []
#     for p in paths:
#         if not p.exists():
#             continue
#         with p.open("r", encoding="utf-8") as f:
#             for line in f:
#                 line = line.strip()
#                 if not line:
#                     continue
#                 rows.append(json.loads(line))
#     return rows

# def load_test_rows(root: Path) -> List[Dict[str, Any]]:
#     paths = []
#     for task in ("predicate", "argument"):
#         base = root / task / "test"
#         paths += [base / "search_and_rescue.jsonl",
#                   base / "warehouse.jsonl",
#                   base / "traffic_light.jsonl"]
#     rows = _read_jsonl(paths)
#     if not rows:
#         raise FileNotFoundError(f"No test data found under {root}.")
#     return rows

# test_rows = load_test_rows(DATA_ROOT)
# len(test_rows), list(test_rows[0].keys())
# -------------------------------
# Data loading
# -------------------------------
def _read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    if not path.exists():
        print(f"[warn] File not found: {path}")
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows
SUITES = ["libero_10", "libero_90", "libero_object", "libero_goal", "libero_spatial"]
LIBERO_DATA_DIR = Path("libero_grounding_data")  # Output of convert_libero_dataset.py

def load_libero_data(data_dir: Path, suites: List[str]) -> List[Dict[str, Any]]:
    """Load LIBERO grounding data from JSONL files."""
    all_rows = []
    for suite in suites:
        path = data_dir / f"{suite}_grounding.jsonl"
        rows = _read_jsonl(path)
        print(f"  {suite}: {len(rows)} entries")
        all_rows.extend(rows)
    return all_rows

print(f"Loading LIBERO data from: {LIBERO_DATA_DIR}")
test_rows = load_libero_data(LIBERO_DATA_DIR, SUITES)
print(f"\nTotal: {len(test_rows)} entries")

if test_rows:
    print(f"Sample entry keys: {list(test_rows[0].keys())}")

In [ ]:

# -------------------------------
# Derive helper fields per row
# -------------------------------
def row_domain(row: Dict[str, Any]) -> str:
    # domain is the first token of the prefix
    return row["prefix"][0] if row.get("prefix") else "<?>"

def row_task(row: Dict[str, Any]) -> str:
    pf = row.get("prefix", [])
    if "<predicates>" in pf:
        return "predicate"
    if "<const>" in pf:
        return "argument"
    return "unknown"

def marker_index(prefix: List[str], marker: str) -> int:
    try:
        return prefix.index(marker)
    except ValueError:
        return -1

# attach task and domain for grouping later
for r in test_rows:
    r["_domain"] = row_domain(r)
    r["_task"] = row_task(r)



In [ ]:

# -------------------------------
# Tokenization on the fly in a collator so we can keep original metadata
# -------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

class GrounderDataset(torch.utils.data.Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx): return self.rows[idx]

def collate(batch: List[Dict[str, Any]]):
    sents = [" ".join(b["sentence"]) if isinstance(b["sentence"], list) else str(b["sentence"]) for b in batch]
    prefixes = [" ".join(b["prefix"]) if isinstance(b["prefix"], list) else str(b["prefix"]) for b in batch]
    enc = tokenizer(sents, prefixes, padding=True, truncation=True, max_length=512, return_tensors="pt")
    labels = [b["prefix_target"] for b in batch]
    # pad labels to MAX_PREFIX_SHARD
    labels = [ (l + [0]*max(0, MAX_PREFIX_SHARD-len(l)))[:MAX_PREFIX_SHARD] for l in labels ]
    enc["labels"] = torch.tensor(labels, dtype=torch.float32)
    # also pass through original objects for analysis
    enc["meta"] = batch
    return enc



In [ ]:

from torch.utils.data import DataLoader
dl = DataLoader(GrounderDataset(test_rows), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

# Run prediction
all_probs, all_preds, all_labels, all_meta = [], [], [], []
with torch.no_grad():
    for enc in dl:
        labels = enc.pop("labels").to(device)
        meta = enc.pop("meta")
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        logits = out.logits
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).to(torch.int)

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        all_meta.extend(meta)

import numpy as np
y_prob = np.vstack(all_probs)
y_pred = np.vstack(all_preds).astype(int)
y_true = np.vstack(all_labels).astype(int)
len(all_meta), y_true.shape, y_pred.shape


In [ ]:

# -------------------------------
# Metrics helpers
# -------------------------------
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import pandas as pd

def overall_metrics(y_true, y_pred):
    pm, rm, fm, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
    pM, rM, fM, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    return {"accuracy": acc, "f1_micro": fm, "f1_macro": fM, "precision_micro": pm, "recall_micro": rm,
            "precision_macro": pM, "recall_macro": rM}

overall = overall_metrics(y_true, y_pred)
pd.Series(overall)


In [ ]:

# -------------------------------
# Per-domain metrics
# -------------------------------
def mask_rows(cond_iter):
    idxs = np.nonzero(np.array(list(cond_iter)))[0]
    return y_true[idxs], y_pred[idxs]

domains = sorted(set(r["_domain"] for r in all_meta))
rows = []
for d in domains:
    y_t_d, y_p_d = mask_rows(r["_domain"] == d for r in all_meta)
    m = overall_metrics(y_t_d, y_p_d)
    m["domain"] = d
    m["n_rows"] = len(y_t_d)
    rows.append(m)

df_domain = pd.DataFrame(rows).set_index("domain").sort_values("f1_micro", ascending=False)
df_domain


In [ ]:

# -------------------------------
# Per-predicate metrics (predicate shards only, tail after "<predicates>")
# -------------------------------
from collections import defaultdict

def per_item_metrics(meta_rows, y_true, y_pred, tail_marker: str) -> pd.DataFrame:
    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int); sup = defaultdict(int)

    for r, yt, yp in zip(meta_rows, y_true, y_pred):
        prefix = r["prefix"]
        if tail_marker not in prefix:
            continue
        m = prefix.index(tail_marker)
        for j in range(m+1, len(prefix)):
            tok = prefix[j]
            yj = int(yt[j]) if j < len(yt) else 0
            pj = int(yp[j]) if j < len(yp) else 0
            sup[tok] += yj
            if yj == 1 and pj == 1: tp[tok] += 1
            elif yj == 0 and pj == 1: fp[tok] += 1
            elif yj == 1 and pj == 0: fn[tok] += 1

    rows = []
    for tok in sorted(set(list(tp) + list(fp) + list(fn) + list(sup))):
        t, f_p, f_n, s = tp[tok], fp[tok], fn[tok], sup[tok]
        prec = t / (t + f_p) if (t + f_p) > 0 else 0.0
        rec  = t / (t + f_n) if (t + f_n) > 0 else 0.0
        f1   = (2*prec*rec / (prec+rec)) if (prec+rec) > 0 else 0.0
        rows.append({"token": tok, "support": int(s), "tp": int(t), "fp": int(f_p), "fn": int(f_n),
                     "precision": prec, "recall": rec, "f1": f1})
    df = pd.DataFrame(rows).sort_values(["f1","support"], ascending=[False, False])
    return df

pred_mask = [("<predicates>" in r["prefix"]) for r in all_meta]
meta_pred = [r for r, m in zip(all_meta, pred_mask) if m]
y_true_pred = y_true[np.array(pred_mask)]
y_pred_pred = y_pred[np.array(pred_mask)]

df_predicate = per_item_metrics(meta_pred, y_true_pred, y_pred_pred, "<predicates>")
df_predicate.head(20)


In [ ]:

# -------------------------------
# Per-constant metrics (argument shards only, tail after "<const>")
# -------------------------------
arg_mask = [("<const>" in r["prefix"]) for r in all_meta]
meta_arg = [r for r, m in zip(all_meta, arg_mask) if m]
y_true_arg = y_true[np.array(arg_mask)]
y_pred_arg = y_pred[np.array(arg_mask)]

df_constant = per_item_metrics(meta_arg, y_true_arg, y_pred_arg, "<const>")
df_constant.head(20)


In [ ]:

# Save artifacts
out_dir = MODEL_DIR / "eval_breakdown"
out_dir.mkdir(parents=True, exist_ok=True)
df_domain.to_csv(out_dir / "per_domain.csv", index=True)
df_predicate.to_csv(out_dir / "per_predicate.csv", index=False)
df_constant.to_csv(out_dir / "per_constant.csv", index=False)

print("Saved:")
print(out_dir / "per_domain.csv")
print(out_dir / "per_predicate.csv")
print(out_dir / "per_constant.csv")


In [ ]:

# ============================================
# Write a JSONL copy of the test set with predictions (no CSVs, no eval fn)
# ============================================
from pathlib import Path
import json
from typing import List, Dict, Any, Union
import pandas as pd

def _load_jsonl(path: Union[str, Path]) -> List[Dict[str, Any]]:
    path = Path(path)
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                rows.append(json.loads(s))
    return rows

def _write_jsonl(path: Union[str, Path], rows: List[Dict[str, Any]]) -> None:
    path = Path(path)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def _as_pred_dict(p):
    """
    Normalize a prediction payload to a dict for storage under 'predictions'.
    Accepts a scalar (wrapped as {'value': scalar}) or a dict (passed through).
    """
    if isinstance(p, dict):
        return p
    return {"value": p}

def attach_predictions_from_dataframe(test_jsonl_path, df: "pd.DataFrame", out_dir, *, id_key_candidates=("id","example_id","idx"), out_suffix="_test_with_predictions.jsonl", pred_prefixes=("pred",), meta_prefixes=("correct","score","prob","conf","logit","loss")):
    """
    Merge predictions from a DataFrame into the original test JSONL.
    Attempts to align by a shared id if present, else by index (requires equal lengths).
    """
    rows = _load_jsonl(test_jsonl_path)
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)

    # Identify id key present both in df and rows
    json_keys = set().union(*[set(r.keys()) for r in rows]) if rows else set()
    id_key = None
    for k in id_key_candidates:
        if (k in df.columns) and (k in json_keys):
            id_key = k
            break

    # Collect pred and meta columns
    pred_cols = [c for c in df.columns if any(str(c).lower().startswith(pfx) for pfx in pred_prefixes)]
    meta_cols = [c for c in df.columns if any(str(c).lower().startswith(pfx) for pfx in meta_prefixes)]

    if id_key:
        df_key = df[[id_key] + pred_cols + meta_cols].copy().fillna("")
        pred_map = {r[id_key]: r for _, r in df_key.iterrows()}
        out_rows = []
        missing = 0
        for r in rows:
            rid = r.get(id_key, None)
            extra = {}
            if rid in pred_map:
                pr = pred_map[rid]
                extra = {
                    "predictions": {c: pr[c] for c in pred_cols},
                    "eval_meta":   {c: pr[c] for c in meta_cols},
                }
            else:
                missing += 1
            out = dict(r); out.update(extra)
            out_rows.append(out)
        if missing:
            print(f"[warn] {missing} rows in {Path(test_jsonl_path).name} had no matching id in DataFrame")
    else:
        if len(df) != len(rows):
            raise ValueError(f"Index-alignment impossible: df={len(df)} vs rows={len(rows)} for {test_jsonl_path}")
        df2 = df.fillna("")
        out_rows = []
        for i, r in enumerate(rows):
            pr = df2.iloc[i]
            extra = {
                "predictions": {c: pr[c] for c in pred_cols},
                "eval_meta":   {c: pr[c] for c in meta_cols},
            }
            out = dict(r); out.update(extra)
            out_rows.append(out)

    out_path = Path(out_dir) / f"{Path(test_jsonl_path).stem}{out_suffix}"
    _write_jsonl(out_path, out_rows)
    print(f"[ok] Wrote: {out_path}")

def attach_predictions_from_list(test_jsonl_path, preds_list: List[Union[Dict[str, Any], Any]], out_dir, *, out_suffix="_test_with_predictions.jsonl"):
    """
    Attach predictions from a Python list aligned by index to each example in the test JSONL.
    Each list item can be a dict or a scalar; scalars are wrapped as {'value': scalar}.
    """
    rows = _load_jsonl(test_jsonl_path)
    if len(preds_list) != len(rows):
        raise ValueError(f"Length mismatch: preds_list={len(preds_list)} vs rows={len(rows)} for {test_jsonl_path}")

    out_rows = []
    for r, p in zip(rows, preds_list):
        out = dict(r)
        out["predictions"] = _as_pred_dict(p)
        out_rows.append(out)

    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    out_path = Path(out_dir) / f"{Path(test_jsonl_path).stem}{out_suffix}"
    _write_jsonl(out_path, out_rows)
    print(f"[ok] Wrote: {out_path}")

def attach_predictions_from_idmap(test_jsonl_path, id_to_pred: Dict[Any, Union[Dict[str, Any], Any]], out_dir, *, id_key="id", out_suffix="_test_with_predictions.jsonl"):
    """
    Attach predictions using a dict keyed by an id present in each JSONL row.
    """
    rows = _load_jsonl(test_jsonl_path)
    out_rows = []
    missing = 0
    for r in rows:
        rid = r.get(id_key, None)
        out = dict(r)
        if rid in id_to_pred:
            out["predictions"] = _as_pred_dict(id_to_pred[rid])
        else:
            missing += 1
        out_rows.append(out)
    if missing:
        print(f"[warn] {missing} rows in {Path(test_jsonl_path).name} had no prediction in id_to_pred (id_key='{id_key}')")

    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    out_path = Path(out_dir) / f"{Path(test_jsonl_path).stem}{out_suffix}"
    _write_jsonl(out_path, out_rows)
    print(f"[ok] Wrote: {out_path}")




